# mko_data_cleaner — pipeline example

`mko_data_cleaner` as one step in a Jupyter pipeline: install (pinned to
the `v0.2.0` tag) → import + version → `initialize_settings` → create a
tiny synthetic Mediascope-style report (uppercase headers with DELETE /
REPLACE / ADD dictionary rules) → `process_data` → consume the resulting
`ReportResult` and read the generated `clean_data` and `null_data` gzip CSVs.

The whole sample is generated in-cell into a temporary directory, so no data
artifacts are committed to the repository.

In [ ]:
try:
    import mko_data_cleaner  # noqa: F401
except ImportError:
    get_ipython().run_line_magic(
        "pip",
        'install "git+https://github.com/manicko/mko_data_cleaner.git@v0.2.0"',
    )  # noqa: E501
# On a clean kernel this installs the pinned v0.2.0 tag (T7).
# In the dev venv (package already installed) this cell is a no-op, so the
# downstream cells run without network access.

## Import the library and public API

`mko_data_cleaner` exposes `initialize_settings`, `process_data`,
`ReportResult`, `SettingsNotInitializedError`, and `__version__` at the top
level.

In [ ]:
import tempfile
from pathlib import Path

import polars as pl
from mko_data_cleaner import initialize_settings, process_data, ReportResult
import mko_data_cleaner

print("mko_data_cleaner", mko_data_cleaner.__version__)
print("process_data ->", process_data.__doc__.strip().splitlines()[0])

## Initialize settings

`initialize_settings` copies the shipped `app_config.yaml` and
`log_config.yaml` templates into `settings/` under the given user
directory and returns the path to that `settings/` directory. The
pipeline reads its configuration from there.

In [ ]:
settings_dir = Path(tempfile.mkdtemp())
initialized = initialize_settings(settings_dir=settings_dir)
print("settings written to:", initialized)
for p in sorted(initialized.iterdir()):
    print(" -", p.name)

In [ ]:
import yaml

cfg = yaml.safe_load((initialized / "app_config.yaml").read_text())
cfg["data_file_settings"]["index_column"] = "ADID"
# The sample uses the README uppercase header family (ADID), so override
# index_column to ADID to match it deterministically (the shipped template
# uses the camelCase "AdId").
# NOTE: the model field `date_column: str = None` is not Optional, so an
# explicit YAML null raises a ValidationError. This sample has no date
# column, so we DROP the key to let the default None apply instead of
# setting it to None.
cfg["data_file_settings"].pop("date_column", None)
(initialized / "app_config.yaml").write_text(yaml.safe_dump(cfg))
print("index_column ->", cfg["data_file_settings"]["index_column"])
print("date_column dropped (default None -> no date column detected)")

## Expected report directory layout

```
report_dir/
  raw_data/<*.gz>         # gzipped data CSVs (extension from config)
  dict/merged_dictionary.csv   # plain merged dictionary (read directly)
  clean_data/             # created at run time: data_table_*.csv.gz
  data_base/db_example.db # created at run time (SQLite)
  null_data_*.csv.gz      # created at run time: rows with no dictionary match
```

In [ ]:
for p in ("raw_data", "dict", "clean_data", "data_base"):
    print(f"  {p}/")
print("  null_data_*.csv.gz  (created after the run)")

## Create a tiny synthetic report (DELETE / REPLACE / ADD)

Seven rows with uppercase Mediascope-family headers. The dictionary exercises all
three rule actions targeting the `BRANDS` column (data-table index 3):

| action | term    | extra `tag` value | effect on matching rows |
|--------|---------|-------------------|-------------------------|
| d      | Brand2  | —                 | delete the row          |
| r      | Apple   | fruit             | write `tag` = fruit     |
| a      | Brand1  | added_by_brand    | append `tag` value      |

In [ ]:
import gzip

report_dir = Path(tempfile.mkdtemp())
raw = report_dir / "raw_data"
raw.mkdir(parents=True)
ddir = report_dir / "dict"
ddir.mkdir(parents=True)

header = "YEAR;MONTH;ADVERTISERS;BRANDS;ADID;MEDIA;COST_RUB;QUANTITY"
data_rows = [
    "2024;01;EXAMPLE;Apple;id1;TV;1500000;25",
    "2024;01;EXAMPLE;Brand2;id2;TV;3200000;12",
    "2024;02;FAKE;Brand1;id3;TV;450000;8",
    "2024;02;FAKE;Orange;id4;TV;100000;5",
    "2024;03;EXAMPLE;Apple;id5;TV;200000;10",
    "2024;03;FAKE;Pear;id6;TV;150000;3",
    "2024;03;EXAMPLE;Brand2;id7;TV;500000;7",
]
with gzip.open(raw / "sample.csv.gz", "wt", encoding="utf-8") as f:
    f.write(header + "\n" + "\n".join(data_rows) + "\n")

dict_header = "action;match;search;value;term;tag"
dict_rows = [
    "d;f;3;NULL;Brand2;",
    "r;f;3;NULL;Apple;fruit",
    "a;f;3;NULL;Brand1;added_by_brand",
]
(ddir / "merged_dictionary.csv").write_text(
    dict_header + "\n" + "\n".join(dict_rows) + "\n", encoding="utf-8"
)

print(pl.read_csv(raw / "sample.csv.gz", separator=";").head())
print(pl.read_csv(ddir / "merged_dictionary.csv", separator=";"))

## Run the pipeline

In [ ]:
result: ReportResult = process_data(report_dir, settings_dir=settings_dir)
print(result)

## Inspect the result

`ReportResult` exposes `rows_loaded`, `rows_exported`, `timings` (`{import, mapping, export}`), `export_dir`, `null_data_files`, `import_dir`, `report_dir`, `db_path`, plus timing/duration fields.

In [ ]:
print("rows_loaded:", result.rows_loaded)
print("rows_exported:", result.rows_exported)
print("timings:", result.timings)
print("export_dir:", result.export_dir)
print("null_data_files:", result.null_data_files)
clean = next(result.export_dir.glob("*.csv.gz"))
print("clean_data sample:"); display(pl.read_csv(clean, separator=";").head())
if result.null_data_files:
    nd = result.null_data_files[0]
    print("null_data sample:"); display(pl.read_csv(nd, separator=";").head())

## Notes & troubleshooting

- **Pinning (D1/D11):** the install cell pins `git+https://...mko_data_cleaner.git@v0.2.0`. That tag is cut in **T7**; at B5 time it does not exist. In the dev venv the import guard makes the install cell a no-op (package is already installed), so you can `Run all` without network access.
- **Temporary dirs (C5):** settings and report dirs live in `tempfile.mkdtemp()` per run, so re-runs are idempotent and the repo stays clean.
- **Config override:** `index_column` is set to `ADID` to match the uppercase sample; `date_column` is dropped (not `None`) because the Pydantic field is `str`, not `Optional[str]` — an explicit YAML null raises a `ValidationError`.
- **`SettingsNotInitializedError`:** run `initialize_settings()` first (cell 6).
- **Counts:** see the README for the verified row counts for this sample.
- No committed output cells — outputs are generated at run time.